In [4]:
from google.colab import drive
drive.mount('/content/drive')

In [5]:
!nvidia-smi

In [3]:
!pip install -q --upgrade transformers datasets
!pip install -q --upgrade accelerate
!pip install -q evaluate huggingface_hub pyarrow

print("Packages installed.")

In [6]:
import os
import json
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
from torch.utils.data import DataLoader

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding
)
from datasets import Dataset, DatasetDict
import evaluate

from sklearn.metrics import (
    classification_report, confusion_matrix,
    f1_score, precision_score, recall_score, accuracy_score
)
from sklearn.preprocessing import LabelEncoder

warnings.filterwarnings('ignore')
sns.set_style("whitegrid")

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'No GPU'}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

In [7]:
drive_data_path = Path("/content/drive/MyDrive/review-intelligence-system/data/processed")

if drive_data_path.exists():
    print(f"Found data folder: {drive_data_path}")
    for f in drive_data_path.iterdir():
        size_mb = f.stat().st_size / 1024**2
        print(f"  {f.name}: {size_mb:.1f} MB")
else:
    print(f"Folder not found: {drive_data_path}")
    print("Please upload train.parquet, val.parquet, test.parquet to that folder.")

In [8]:
train_df = pd.read_parquet(drive_data_path / "train.parquet")
val_df = pd.read_parquet(drive_data_path / "val.parquet")
test_df = pd.read_parquet(drive_data_path / "test.parquet")

print(f"Train: {len(train_df):,}")
print(f"Val:   {len(val_df):,}")
print(f"Test:  {len(test_df):,}")
print(f"\nColumns: {train_df.columns.tolist()}")
train_df.head(2)

In [9]:
label_encoder = LabelEncoder()
y_train = label_encoder.fit_transform(train_df['sentiment'])
y_val = label_encoder.transform(val_df['sentiment'])
y_test = label_encoder.transform(test_df['sentiment'])

label2id = {label: int(i) for i, label in enumerate(label_encoder.classes_)}
id2label = {int(i): label for i, label in enumerate(label_encoder.classes_)}

print(f"Label mapping: {label2id}")
print(f"Number of classes: {len(label_encoder.classes_)}")

In [10]:
def to_hf_dataset(df, labels):
    return Dataset.from_dict({
        'text': df['text_clean'].tolist(),
        'labels': labels.tolist(),
        'category': df['category'].tolist()  # for category-level eval later
    })

train_ds = to_hf_dataset(train_df, y_train)
val_ds = to_hf_dataset(val_df, y_val)
test_ds = to_hf_dataset(test_df, y_test)

datasets_dict = DatasetDict({
    'train': train_ds,
    'val': val_ds,
    'test': test_ds
})

print(datasets_dict)

In [11]:
MODEL_NAME = "distilbert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

sample_text = train_df['text_clean'].iloc[0]
print(f"Sample text:\n{sample_text[:200]}...\n")

tokens = tokenizer(sample_text, truncation=True, max_length=256)
print(f"Tokenized length: {len(tokens['input_ids'])}")
print(f"First 20 token IDs: {tokens['input_ids'][:20]}")
print(f"Decoded back: {tokenizer.decode(tokens['input_ids'][:20])}")

In [12]:
def tokenize_function(examples):
    return tokenizer(
        examples['text'],
        truncation=True,
        max_length=256,
        padding=False
    )


print("Tokenizing datasets...")
start = time.time()

tokenized_datasets = datasets_dict.map(
    tokenize_function,
    batched=True,
    remove_columns=['text', 'category']
)

print(f"Done in {time.time() - start:.1f} seconds")
print(tokenized_datasets)

In [13]:
from sklearn.utils.class_weight import compute_class_weight

class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.array([0, 1, 2]),
    y=y_train
)

print(f"Class weights:")
for cls_idx, weight in enumerate(class_weights):
    print(f"  {label_encoder.classes_[cls_idx]}: {weight:.4f}")

class_weights_tensor = torch.tensor(class_weights, dtype=torch.float).to(device)

In [14]:
from torch import nn

class WeightedTrainer(Trainer):
    """Trainer with weighted cross-entropy loss for class imbalance."""

    def __init__(self, *args, class_weights=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits

        loss_fct = nn.CrossEntropyLoss(weight=self.class_weights)
        loss = loss_fct(logits, labels)

        return (loss, outputs) if return_outputs else loss

In [15]:
def compute_metrics(eval_pred):
    """Compute multiple metrics during training."""
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)

    return {
        'accuracy': accuracy_score(labels, predictions),
        'f1_macro': f1_score(labels, predictions, average='macro'),
        'f1_weighted': f1_score(labels, predictions, average='weighted'),
        'f1_negative': f1_score(labels, predictions, labels=[0], average='macro'),
        'f1_neutral': f1_score(labels, predictions, labels=[1], average='macro'),
        'f1_positive': f1_score(labels, predictions, labels=[2], average='macro'),
    }

In [16]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=3,
    label2id=label2id,
    id2label=id2label
)

model = model.to(device)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameters:     {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"Model size: ~{total_params * 4 / 1024**2:.0f} MB (fp32)")

In [17]:
output_dir = "/content/drive/MyDrive/review-intelligence-system/models/distilbert"

training_args = TrainingArguments(
    output_dir=output_dir,
    num_train_epochs=3,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_ratio=0.1,

    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,

    logging_dir="/content/drive/MyDrive/review-intelligence-system/logs",
    logging_steps=100,
    report_to="none",  # disable wandb/tensorboard auto-logging

    fp16=True,  # mixed precision for T4
    gradient_accumulation_steps=1,

    save_total_limit=2,

    seed=RANDOM_SEED,
)

print("Training arguments configured.")

In [18]:
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets['train'],
    eval_dataset=tokenized_datasets['val'],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    class_weights=class_weights_tensor
)

print("Trainer initialized.")

In [19]:
print("Starting training...")
print(f"Total examples: {len(tokenized_datasets['train']):,}")
print(f"Batch size: 32")
print(f"Steps per epoch: ~{len(tokenized_datasets['train']) // 32:,}")
print(f"Total epochs: 3")
print()

start = time.time()
train_result = trainer.train()
training_time = time.time() - start

print(f"\nTraining completed in {training_time / 60:.1f} minutes")
print(f"Final training loss: {train_result.training_loss:.4f}")

In [20]:
# Final validation evaluation with best model
val_results = trainer.evaluate(tokenized_datasets['val'])
print("Validation results (best model):")
for key, value in val_results.items():
    if isinstance(value, float):
        print(f"  {key}: {value:.4f}")

In [21]:
print("Evaluating on test set...")
test_results = trainer.evaluate(tokenized_datasets['test'])
print("\nTest results:")
for key, value in test_results.items():
    if isinstance(value, float):
        print(f"  {key}: {value:.4f}")

In [22]:
print("Generating test predictions...")
test_predictions = trainer.predict(tokenized_datasets['test'])
y_test_pred = np.argmax(test_predictions.predictions, axis=1)
y_test_true = test_predictions.label_ids

n_samples = len(y_test_true)
total_time_ms = test_predictions.metrics.get('test_runtime', 0) * 1000
inference_per_sample = total_time_ms / n_samples
print(f"\nInference: {inference_per_sample:.2f} ms/sample (on T4 GPU)")

print("\nClassification Report:")
print(classification_report(
    y_test_true, y_test_pred,
    target_names=label_encoder.classes_,
    digits=4
))

print("\nPer-category macro F1:")
test_categories = test_df['category'].values
for cat in np.unique(test_categories):
    mask = test_categories == cat
    cat_f1 = f1_score(y_test_true[mask], y_test_pred[mask], average='macro')
    print(f"  {cat:<15}: {cat_f1:.4f}")

In [25]:
labels = label_encoder.classes_
cm = confusion_matrix(y_test_true, y_test_pred)
cm_normalized = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=labels, yticklabels=labels, ax=axes[0])
axes[0].set_title('DistilBERT — Confusion Matrix (Counts)', fontweight='bold')
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('True')

sns.heatmap(cm_normalized, annot=True, fmt='.2%', cmap='Blues',
            xticklabels=labels, yticklabels=labels, ax=axes[1])
axes[1].set_title('DistilBERT — Confusion Matrix (Normalized)', fontweight='bold')
axes[1].set_xlabel('Predicted')
axes[1].set_ylabel('True')

plt.tight_layout()
plt.savefig('/content/drive/MyDrive/review-intelligence-system/reports/figures/09_distilbert_confusion.png', dpi=100, bbox_inches='tight')
plt.show()

In [26]:
# Save best model + tokenizer
final_model_path = "/content/drive/MyDrive/review-intelligence-system/models/distilbert-final"
trainer.save_model(final_model_path)
tokenizer.save_pretrained(final_model_path)

print(f"Model saved to: {final_model_path}")

# Show what's in there
import os
for f in os.listdir(final_model_path):
    size_mb = os.path.getsize(os.path.join(final_model_path, f)) / 1024**2
    print(f"  {f}: {size_mb:.1f} MB")

In [27]:
distilbert_results = {
    'model_name': 'DistilBERT',
    'val': {k.replace('eval_', ''): v for k, v in val_results.items() if isinstance(v, float)},
    'test': {k.replace('eval_', ''): v for k, v in test_results.items() if isinstance(v, float)},
    'training_time_minutes': training_time / 60,
    'inference_ms_per_sample': inference_per_sample,
    'total_parameters': total_params,
    'config': {
        'model': MODEL_NAME,
        'max_length': 256,
        'batch_size': 32,
        'learning_rate': 2e-5,
        'epochs': 3,
        'class_weights': class_weights.tolist()
    }
}

results_path = "/content/drive/MyDrive/review-intelligence-system/distilbert_results.json"
with open(results_path, 'w') as f:
    json.dump(distilbert_results, f, indent=2)

print(f"Results saved: {results_path}")
print(f"\nKey metrics (test set):")
print(f"  Macro F1:     {distilbert_results['test']['f1_macro']:.4f}")
print(f"  Accuracy:     {distilbert_results['test']['accuracy']:.4f}")
print(f"  Negative F1:  {distilbert_results['test']['f1_negative']:.4f}")
print(f"  Neutral F1:   {distilbert_results['test']['f1_neutral']:.4f}")
print(f"  Positive F1:  {distilbert_results['test']['f1_positive']:.4f}")

In [29]:
from huggingface_hub import login

# Login with your token
login(token="hf_xxxxxxxxxxxxxxxxxxxxxxxxx")

# Push to hub
model.push_to_hub("review-sentiment-distilbert", private=False)
tokenizer.push_to_hub("review-sentiment-distilbert", private=False)

print("Model pushed to: https://huggingface.co/8rav0/review-sentiment-distilbert")